# SLAYER-OCR: RF-DETR na poprawionym GT v2

Wybierz **Runtime > Change runtime type > GPU**, a potem **Runtime > Run all**. Notebook mierzy ten sam zamrożony checkpoint na poprawionych etykietach 12 stron. To nowy pomiar, a nie próba odtworzenia starego `mAP=0,3273`.

In [ ]:
%pip install -q --no-cache-dir "rfdetr[train]==1.11.0"

In [ ]:
import hashlib
import importlib.metadata
import json
from pathlib import Path, PurePosixPath
import shutil
import subprocess
import sys
import zipfile

from google.colab import files
import torch

CODE_REVISION = '964691bd1ea1db7d51abfee30ec50e41628f83ca'
MODEL_NAME = 'slayer-rfdetr-layout-v1-model.zip'
MODEL_SHA256 = '68578a6e008788fd41b6b9f64d31b3a9def82f25c615ead8fc97da5a9962fb0b'
EVIDENCE_NAME = 'slayer-rfdetr-layout-v1-evidence.zip'
EVIDENCE_SHA256 = 'ce5e68096866ab903d7722a75da1b0b5cf44215d597da49c7a8137859773fbde'

assert torch.cuda.is_available(), 'Wybierz runtime GPU i uruchom notebook od początku.'
assert importlib.metadata.version('rfdetr') == '1.11.0'
print('GPU:', torch.cuda.get_device_name(0), 'Torch:', torch.__version__)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def save_uploads(uploaded):
    paths = []
    for name, payload in uploaded.items():
        path = Path('/content') / Path(name).name
        path.write_bytes(payload)
        paths.append(path)
    return paths

def upload_expected(label, expected_name, expected_sha256):
    target = Path('/content') / expected_name
    if target.is_file() and sha256(target) == expected_sha256:
        print(label, 'CACHED_VERIFIED', target)
        return target
    print(f'Wybierz {expected_name}. Możesz wskazać także inne ZIP-y; zostaną rozpoznane po SHA-256.')
    candidates = save_uploads(files.upload())
    for candidate in candidates:
        if sha256(candidate) == expected_sha256:
            if candidate != target:
                shutil.copyfile(candidate, target)
            print(label, 'UPLOAD_VERIFIED', target)
            return target
    raise AssertionError(f'Nie wgrano wymaganego pliku {expected_name}.')

def is_corrected_dataset(path):
    try:
        with zipfile.ZipFile(path) as archive:
            matches = [name for name in archive.namelist()
                       if PurePosixPath(name).name == 'provenance.json']
            if len(matches) != 1:
                return False
            provenance = json.loads(archive.read(matches[0]))
            return provenance.get('schema') == 'slayer-layout-rfdetr-dataset-v2'
    except (OSError, zipfile.BadZipFile, json.JSONDecodeError):
        return False

def upload_corrected_dataset():
    target = Path('/content/slayer-layout-rfdetr-corrected-v2.zip')
    if target.is_file() and is_corrected_dataset(target):
        print('dataset CACHED_VERIFIED', target, sha256(target))
        return target, sha256(target)
    print('Wybierz ZIP utworzony przez training.apply_rfdetr_layout_gt_review.')
    candidates = save_uploads(files.upload())
    matches = [path for path in candidates if is_corrected_dataset(path)]
    assert len(matches) == 1, 'Wgrane pliki muszą zawierać dokładnie jeden dataset v2.'
    if matches[0] != target:
        shutil.copyfile(matches[0], target)
    digest = sha256(target)
    print('dataset UPLOAD_VERIFIED', target, digest)
    return target, digest

## 1/3: poprawiony dataset v2
Wybierz ZIP wygenerowany po zakończonym review. Nazwa może zawierać datę; notebook sprawdzi schemat wewnątrz archiwum.

In [ ]:
dataset_archive, dataset_sha256 = upload_corrected_dataset()

## 2/3: zamrożony model
Wybierz `slayer-rfdetr-layout-v1-model.zip`.

In [ ]:
model_archive = upload_expected('model', MODEL_NAME, MODEL_SHA256)

## 3/3: dowody treningowe
Wybierz `slayer-rfdetr-layout-v1-evidence.zip`.

In [ ]:
evidence_archive = upload_expected('evidence', EVIDENCE_NAME, EVIDENCE_SHA256)

In [ ]:
repo = Path('/content/OCR_engine-corrected-gt')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/PiotrStyla/OCR_engine.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', CODE_REVISION], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', CODE_REVISION], check=True)
resolved = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert resolved == CODE_REVISION, (resolved, CODE_REVISION)
sys.path.insert(0, str(repo))
print('CODE_VERIFIED', resolved)

In [ ]:
from training.audit_slayer_rfdetr_layout import run

result_archive, report = run(
    dataset_archive=dataset_archive,
    model_archive=model_archive,
    evidence_archive=evidence_archive,
    expected_hashes={
        'dataset': dataset_sha256,
        'model': MODEL_SHA256,
        'evidence': EVIDENCE_SHA256,
    },
    output_root='/content',
    code_revision=CODE_REVISION,
    prediction_threshold=0.05,
    audit_threshold=0.25,
)
assert report['audit_mode'] == 'corrected-gt-reanalysis'
assert report['release_status'] == 'public-experiment-evidence'
print(json.dumps(report, ensure_ascii=False, indent=2))
print('EVIDENCE_READY', result_archive, sha256(result_archive))

## Pobranie wyniku
Pobierz ZIP dowodowy. Będzie zawierał nowy wynik, predykcje, nakładki, kolejkę hard examples, provenance i sumy kontrolne.

In [ ]:
files.download(str(result_archive))